In [1]:
import sys
print(sys.executable)

import torch
print(torch.__version__)

D:\Anaconda\envs\comp6713\python.exe
2.11.0+cu126


In [2]:
# Fix local environment issues before importing transformers.

import os
import sys
import site

# Do not use TensorFlow / Flax.
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

# Remove user-site packages from sys.path.
# This helps avoid loading a wrong torchvision from:
# C:\Users\...\AppData\Roaming\Python\Python310\site-packages
user_site = site.getusersitepackages()
if user_site in sys.path:
    sys.path.remove(user_site)

print("Removed user site-packages from sys.path:")
print(user_site)

Removed user site-packages from sys.path:
C:\Users\千秋凛然\AppData\Roaming\Python\Python310\site-packages


In [3]:
# =========================
# Install packages if needed
# =========================
# !pip install sympy safetensors gradio transformers joblib scikit-learn pandas

In [4]:
# =========================
# Import libraries & Set project root
# =========================
import re
from pathlib import Path

import gradio as gr
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from safetensors.torch import load_file
from scipy.sparse import hstack
from transformers import AutoConfig, AutoModel, AutoModelForSequenceClassification, AutoTokenizer

# Find the project folder.
# This notebook should be inside 04_notebooks.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "04_notebooks" else NOTEBOOK_DIR

DATA_DIR = PROJECT_ROOT / "00_shared_data" / "processed"
LEXICON_PATH = PROJECT_ROOT / "00_shared_data" / "lexicon" / "CrisisLexRec.txt"
MODELS_DIR = PROJECT_ROOT / "01_saved_models"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DEVICE =", DEVICE)

PROJECT_ROOT = E:\Disaster_Tweet_Project
DEVICE = cuda


In [5]:
# Fixed label order used in this project.
LABELS = [
    "caution_and_advice",
    "displaced_people_and_evacuations",
    "donation_and_volunteering",
    "infrastructure_and_utilities_damage",
    "injured_or_dead_people",
    "missing_trapped_or_found_people",
    "not_related_or_irrelevant",
    "other_useful_information",
    "sympathy_and_emotional_support",
]

LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

In [6]:
# Read the lexicon file.
def load_lexicon(path):
    terms = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            term = line.strip().lower()
            if term:
                terms.append(term)

    unigram_terms = {term for term in terms if " " not in term}
    phrase_terms = [term for term in terms if " " in term]
    return terms, unigram_terms, phrase_terms


LEXICON_TERMS, LEXICON_UNIGRAMS, LEXICON_PHRASES = load_lexicon(LEXICON_PATH)


def extract_lexicon_features(text):
    lowered = str(text).lower()
    tokens = re.findall(r"\b\w+\b", lowered)
    token_count = max(len(tokens), 1)

    matched_unigrams = {token for token in tokens if token in LEXICON_UNIGRAMS}
    matched_phrases = [term for term in LEXICON_PHRASES if term in lowered]
    unique_terms = matched_unigrams | set(matched_phrases)

    return np.array([
        float(len(matched_unigrams)),
        float(len(matched_phrases)),
        float(len(unique_terms)),
        float(len(unique_terms) / token_count),
        float(bool(unique_terms)),
    ], dtype=np.float32)


# Baseline uses a simpler lexicon count.
def count_lexicon_hits_for_baseline(text):
    words = str(text).lower().split()
    return sum(1 for w in words if w in LEXICON_UNIGRAMS)

In [7]:
# Custom BERT model used in member 4 experiments.
# It combines BERT text features with lexicon features.
class BertWithLexiconFeatures(nn.Module):
    def __init__(self, model_dir, num_labels=9, feature_dim=5):
        super().__init__()
        config = AutoConfig.from_pretrained(model_dir)
        self.encoder = AutoModel.from_pretrained(model_dir, config=config)
        dropout_prob = getattr(config, "hidden_dropout_prob", 0.1)
        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(config.hidden_size + feature_dim, num_labels)

    def forward(self, input_ids, attention_mask, lexicon_features):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)

        if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            pooled = outputs.pooler_output
        else:
            pooled = outputs.last_hidden_state[:, 0, :]

        pooled = self.dropout(pooled)
        combined = torch.cat([pooled, lexicon_features], dim=1)
        logits = self.classifier(combined)
        return logits


def load_custom_bert_model(model_path):
    model = BertWithLexiconFeatures(model_path, num_labels=len(LABELS), feature_dim=5)
    state_dict = load_file(str(model_path / "model.safetensors"))
    model.load_state_dict(state_dict, strict=False)

    tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
    model.to(DEVICE)
    model.eval()
    return tokenizer, model

In [8]:
# Load all 7 models.

# 1) Baseline
baseline_package = joblib.load(
    MODELS_DIR / "baseline_tfidf_logreg_lexicon" / "complete_baseline_model.pkl"
)
baseline_vectorizer = baseline_package["vectorizer"]
baseline_classifier = baseline_package["classifier"]

# 2) DistilBERT
distilbert_tokenizer = AutoTokenizer.from_pretrained(
    MODELS_DIR / "distilbert_base_uncased" / "best_model",
    use_fast=True
)
distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    MODELS_DIR / "distilbert_base_uncased" / "best_model"
)
distilbert_model.to(DEVICE)
distilbert_model.eval()

# 3) RoBERTa
roberta_tokenizer = AutoTokenizer.from_pretrained(
    MODELS_DIR / "roberta_base" / "best_model",
    use_fast=True
)
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    MODELS_DIR / "roberta_base" / "best_model"
)
roberta_model.to(DEVICE)
roberta_model.eval()

# 4) BERT + Lexicon (Cross Entropy)
bert_ce_tokenizer, bert_ce_model = load_custom_bert_model(
    MODELS_DIR / "bert_lexicon_cross_entropy" / "best_model"
)

# 5) BERT + Lexicon (Weighted Cross Entropy)
bert_wce_tokenizer, bert_wce_model = load_custom_bert_model(
    MODELS_DIR / "bert_lexicon_weighted_cross_entropy" / "best_model"
)

# 6) BERT + Lexicon (Label Smoothing)
bert_ls_tokenizer, bert_ls_model = load_custom_bert_model(
    MODELS_DIR / "bert_lexicon_label_smoothing" / "best_model"
)

# 7) BERT + Lexicon (Weighted + Label Smoothing)
bert_wls_tokenizer, bert_wls_model = load_custom_bert_model(
    MODELS_DIR / "bert_lexicon_weighted_label_smoothing" / "best_model"
)

print("All models loaded.")

D:\Anaconda\envs\comp6713\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
D:\Anaconda\envs\comp6713\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
D:\Anaconda\envs\comp6713\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.0.2 when using version 1.7.2. This might lead t

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 0it [00:00, ?it/s]

BertModel LOAD REPORT from: E:\Disaster_Tweet_Project\01_saved_models\bert_lexicon_cross_entropy\best_model
Key                                                              | Status     | 
-----------------------------------------------------------------+------------+-
encoder.encoder.layer.{0...11}.attention.output.dense.bias       | UNEXPECTED | 
encoder.encoder.layer.{0...11}.attention.self.value.bias         | UNEXPECTED | 
encoder.encoder.layer.{0...11}.output.LayerNorm.bias             | UNEXPECTED | 
encoder.encoder.layer.{0...11}.attention.self.key.bias           | UNEXPECTED | 
encoder.encoder.layer.{0...11}.attention.self.query.bias         | UNEXPECTED | 
encoder.encoder.layer.{0...11}.intermediate.dense.bias           | UNEXPECTED | 
encoder.encoder.layer.{0...11}.intermediate.dense.weight         | UNEXPECTED | 
encoder.encoder.layer.{0...11}.attention.output.LayerNorm.weight | UNEXPECTED | 
encoder.encoder.layer.{0...11}.attention.output.dense.weight     | UNEXPECTED | 
e

...

Loading weights: 0it [00:00, ?it/s]

BertModel LOAD REPORT from: E:\Disaster_Tweet_Project\01_saved_models\bert_lexicon_label_smoothing\best_model
Key                                                              | Status     | 
-----------------------------------------------------------------+------------+-
encoder.encoder.layer.{0...11}.attention.output.dense.bias       | UNEXPECTED | 
encoder.encoder.layer.{0...11}.attention.self.value.bias         | UNEXPECTED | 
encoder.encoder.layer.{0...11}.output.LayerNorm.bias             | UNEXPECTED | 
encoder.encoder.layer.{0...11}.attention.self.key.bias           | UNEXPECTED | 
encoder.encoder.layer.{0...11}.attention.self.query.bias         | UNEXPECTED | 
encoder.encoder.layer.{0...11}.intermediate.dense.bias           | UNEXPECTED | 
encoder.encoder.layer.{0...11}.intermediate.dense.weight         | UNEXPECTED | 
encoder.encoder.layer.{0...11}.attention.output.LayerNorm.weight | UNEXPECTED | 
encoder.encoder.layer.{0...11}.attention.output.dense.weight     | UNEXPECTED | 

Loading weights: 0it [00:00, ?it/s]

BertModel LOAD REPORT from: E:\Disaster_Tweet_Project\01_saved_models\bert_lexicon_weighted_label_smoothing\best_model
Key                                                              | Status     | 
-----------------------------------------------------------------+------------+-
encoder.encoder.layer.{0...11}.attention.output.dense.bias       | UNEXPECTED | 
encoder.encoder.layer.{0...11}.attention.self.value.bias         | UNEXPECTED | 
encoder.encoder.layer.{0...11}.output.LayerNorm.bias             | UNEXPECTED | 
encoder.encoder.layer.{0...11}.attention.self.key.bias           | UNEXPECTED | 
encoder.encoder.layer.{0...11}.attention.self.query.bias         | UNEXPECTED | 
encoder.encoder.layer.{0...11}.intermediate.dense.bias           | UNEXPECTED | 
encoder.encoder.layer.{0...11}.intermediate.dense.weight         | UNEXPECTED | 
encoder.encoder.layer.{0...11}.attention.output.LayerNorm.weight | UNEXPECTED | 
encoder.encoder.layer.{0...11}.attention.output.dense.weight     | UNEX

All models loaded.


In [9]:
# Prediction helpers

def predict_baseline(text):
    text = str(text)
    x_tfidf = baseline_vectorizer.transform([text])
    lex_feat = np.array([[count_lexicon_hits_for_baseline(text)]], dtype=np.float32)
    x = hstack([x_tfidf, lex_feat])
    pred = baseline_classifier.predict(x)[0]
    return str(pred)


def predict_transformer(text, tokenizer, model, max_length=128):
    encoded = tokenizer(
        str(text),
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt"
    )
    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)
        pred_id = int(torch.argmax(outputs.logits, dim=-1).cpu().item())

    if getattr(model.config, "id2label", None):
        return model.config.id2label[pred_id]
    return ID_TO_LABEL[pred_id]


def predict_custom_bert(text, tokenizer, model, max_length=128):
    encoded = tokenizer(
        str(text),
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt"
    )

    lexicon_features = torch.tensor(
        [extract_lexicon_features(text)],
        dtype=torch.float32
    )

    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
    lexicon_features = lexicon_features.to(DEVICE)

    with torch.no_grad():
        logits = model(
            input_ids=encoded["input_ids"],
            attention_mask=encoded["attention_mask"],
            lexicon_features=lexicon_features,
        )
        pred_id = int(torch.argmax(logits, dim=-1).cpu().item())

    return ID_TO_LABEL[pred_id]

In [10]:
# Main function for the button

def predict_all_models(text):
    text = str(text).strip()

    if not text:
        return pd.DataFrame({
            "Model": [
                "Baseline",
                "DistilBERT",
                "RoBERTa",
                "BERT + Lexicon (CE)",
                "BERT + Lexicon (Weighted CE)",
                "BERT + Lexicon (Label Smoothing)",
                "BERT + Lexicon (Weighted + LS)"
            ],
            "Prediction": ["Please enter text first."] * 7
        })

    rows = [
        ["Baseline", predict_baseline(text)],
        ["DistilBERT", predict_transformer(text, distilbert_tokenizer, distilbert_model)],
        ["RoBERTa", predict_transformer(text, roberta_tokenizer, roberta_model)],
        ["BERT + Lexicon (CE)", predict_custom_bert(text, bert_ce_tokenizer, bert_ce_model)],
        ["BERT + Lexicon (Weighted CE)", predict_custom_bert(text, bert_wce_tokenizer, bert_wce_model)],
        ["BERT + Lexicon (Label Smoothing)", predict_custom_bert(text, bert_ls_tokenizer, bert_ls_model)],
        ["BERT + Lexicon (Weighted + LS)", predict_custom_bert(text, bert_wls_tokenizer, bert_wls_model)],
    ]

    return pd.DataFrame(rows, columns=["Model", "Prediction"])

In [11]:
# Build the Gradio page

with gr.Blocks(title="Disaster Tweet Classification Demo") as demo:
    gr.Markdown("# Disaster Tweet Classification Demo")
    gr.Markdown(
        "Type one tweet. Click the button once. "
        "The page will show results from all 7 models."
    )

    with gr.Row():
        with gr.Column(scale=3):
            text_input = gr.Textbox(
                label="Tweet Text",
                placeholder="Example: Need food and water at the shelter near the flooded bridge.",
                lines=6,
            )
            predict_button = gr.Button("Classify with 7 Models", variant="primary")

        with gr.Column(scale=2):
            output_table = gr.Dataframe(
                headers=["Model", "Prediction"],
                datatype=["str", "str"],
                row_count=(7, "fixed"),
                col_count=(2, "fixed"),
                wrap=True,
                label="Model Results"
            )

    predict_button.click(
        fn=predict_all_models,
        inputs=text_input,
        outputs=output_table
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
